# Étape 1.3 — Où part le temps dans le PSO ? (issue #2)

**Cette version est prête à exécuter : tu n'as rien à modifier.** Exécute les cellules dans l'ordre avec **Maj+Entrée** et lis le texte au-dessus de chacune.

**Avant tout :** Exécution → Modifier le type d'exécution → **T4 GPU**.

**La question qu'on se pose :** le PSO tourne en ~3 s. Est-ce que ce temps est passé à **calculer sur le GPU** (ce qu'on veut), ou à **autre chose** (transferts, travail du CPU) ?

Pour répondre, on découpe une itération de la boucle en 5 morceaux et on chronomètre chacun :

| # | Morceau | Qui travaille |
|---|---|---|
| 1 | `kernelUpdateParticle` : déplacer les particules | GPU |
| 2 | `kernelUpdatePBest` : mettre à jour les meilleurs personnels | GPU |
| 3 | copier tous les pBest du GPU vers le CPU | bus PCIe |
| 4 | boucle qui cherche le meilleur global gBest | CPU |
| 5 | renvoyer gBest du CPU vers le GPU | bus PCIe |

### Cellule 1 — Récupérer le dépôt et se placer sur ta branche

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
token   = userdata.get("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO} || (cd /content/{REPO} && git fetch -q)
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout amina/1.3-chronometrage 2>/dev/null || git checkout -b amina/1.3-chronometrage
!git pull --rebase -q origin amina/1.3-chronometrage 2>/dev/null; git branch --show-current

La dernière ligne doit afficher **`amina/1.3-chronometrage`** : tu travailles sur ta branche, pas sur `main`.

### Cellule 2 — Préparer le dossier de travail

On copie les fichiers du PSO original dans `src/pso_timing/`. On ne touche **jamais** à `src/pso_original/`, qui reste notre référence.

In [ ]:
!mkdir -p /content/DE_CUDA/src/pso_timing
!cp /content/DE_CUDA/src/pso_original/kernel.h /content/DE_CUDA/src/pso_original/kernel.cpp /content/DE_CUDA/src/pso_original/main.cpp /content/DE_CUDA/src/pso_timing/
!ls /content/DE_CUDA/src/pso_timing

### Cellule 3 — Le `kernel.cu` chronométré

C'est le même `kernel.cu` que le prof, avec **un seul changement : des chronomètres dans la boucle**. Tout le reste est identique.

**Comment on chronomètre un morceau GPU (morceaux 1, 2, 3, 5)** — lis ces 4 lignes, c'est le cœur de l'exercice :

```cpp
cudaEventRecord(e0);                  // on pose un « top départ » dans la file du GPU
kernelUpdateParticle<<<...>>>(...);   // le travail à mesurer
cudaEventRecord(e1);                  // on pose un « top arrivée »
cudaEventSynchronize(e1);             // le CPU attend que le GPU ait atteint le top arrivée
cudaEventElapsedTime(&ms, e0, e1);    // temps entre les deux tops, en ms
t_k1 += ms;                           // on additionne sur toutes les itérations
```

**Pourquoi pas une simple horloge CPU autour du kernel ?** Parce qu'un lancement de kernel est **asynchrone** : le CPU lance le kernel et continue *sans attendre*. Une horloge CPU mesurerait seulement le temps de *lancer* (quelques microsecondes), pas le temps de calcul. Les `cudaEvent` sont des repères posés *dans* le GPU, donc ils mesurent le vrai temps GPU.

**Pour le morceau 4 (boucle CPU)**, c'est du code CPU normal, on utilise une horloge CPU : `std::chrono`.

Exécute la cellule : elle doit afficher `Writing …/kernel.cu` (ou `Overwriting`).

In [ ]:
%%writefile /content/DE_CUDA/src/pso_timing/kernel.cu
#include <cuda_runtime.h>
#include <cuda.h>
#include <math_functions.h>

#include "kernel.h"
#include <chrono>


__device__ float tempParticle1[NUM_OF_DIMENSIONS];
__device__ float tempParticle2[NUM_OF_DIMENSIONS];

/* Objective function
0: Levy 3-dimensional
1: Shifted Rastigrin's Function
2: Shifted Rosenbrock's Function
3: Shifted Griewank's Function
4: Shifted Sphere's Function
*/
/**
 * Runs on the GPU, called from the GPU.
*/
__device__ float fitness_function(float x[]) {
    float res = 0;
    float somme = 0;
    float produit = 0;

    switch (SELECTED_OBJ_FUNC)  {
        case 0: 
            float y1 = 1 + (x[0] - 1)/4;
            float yn = 1 + (x[NUM_OF_DIMENSIONS-1] - 1)/4;

            res += pow(sin(phi*y1), 2);

            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float y = 1 + (x[i] - 1)/4;
                float yp = 1 + (x[i+1] - 1)/4;
                res += pow(y - 1, 2)*(1 + 10*pow(sin(phi*yp), 2)) + pow(yn - 1, 2);
            }
            break;
        case 1: 
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2) - 10*cos(2*phi*zi) + 10;
            }
            res -= 330;
            break;
        
        case 2:
            for (int i = 0; i < NUM_OF_DIMENSIONS-1; i++) {
                float zi = x[i] - 0 + 1;
                float zip1 = x[i+1] - 0 + 1;
                res += 100 * ( pow(pow(zi, 2) - zip1, 2)) + pow(zi - 1, 2);
            }
            res += 390;
            break;
        case 3:
            for (int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                somme += pow(zi, 2)/4000;
                produit *= cos(zi/pow(i+1, 0.5));
            }
            res = somme - produit + 1 - 180; 
            break;
        case 4:
            for(int i = 0; i < NUM_OF_DIMENSIONS; i++) {
                float zi = x[i] - 0;
                res += pow(zi, 2);
            }
            res -= 450;
            break;
    }

    return res;
}

/**
 * 
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdateParticle(float *positions, float *velocities, 
                                     float *pBests, float *gBest, float r1, 
                                     float r2)
{

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // avoid an out of bound for the array 
    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS)
        return;

    //float rp = getRandomClamped();
    //float rg = getRandomClamped();
    
    float rp = r1; // random weight for personnal =>  computed from @getRandomClamped
    float rg = r2; // random weight for global =>  computed from @getRandomClamped


    // Mise à jour de velocities et positions
    velocities[i] = OMEGA * velocities[i] + 
                    c1 * rp * (pBests[i] - positions[i]) + 
                    c2 * rg * (gBest[i % NUM_OF_DIMENSIONS] - positions[i]);

    // Update posisi particle
    //Mise à jour de la position de la particule courante
    //incrémentant la position de la particule courante avec la vitesse de la particule courante
    positions[i] += velocities[i];
}

/**
 * Runs on the GPU, called from the CPU or the GPU
*/
__global__ void kernelUpdatePBest(float *positions, float *pBests, float* gBest)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    
    if(i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS || i % NUM_OF_DIMENSIONS != 0)
        return;

    for (int j = 0; j < NUM_OF_DIMENSIONS; j++)
    {
        tempParticle1[j] = positions[i + j];
        tempParticle2[j] = pBests[i + j];
    }

    if (fitness_function(tempParticle1) < fitness_function(tempParticle2))
    {
        for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
            pBests[i + k] = positions[i + k];
    }
}


extern "C" void cuda_pso(float *positions, float *velocities, float *pBests, float *gBest)
{
    int size = NUM_OF_PARTICLES * NUM_OF_DIMENSIONS;
    float *devPos, *devVel, *devPBest, *devGBest;
    float temp[NUM_OF_DIMENSIONS];

    cudaMalloc((void**)&devPos, sizeof(float) * size);
    cudaMalloc((void**)&devVel, sizeof(float) * size);
    cudaMalloc((void**)&devPBest, sizeof(float) * size);
    cudaMalloc((void**)&devGBest, sizeof(float) * NUM_OF_DIMENSIONS);

    int threadsNum = 32;
    int blocksNum = ceil(size / threadsNum);

    cudaMemcpy(devPos, positions, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devVel, velocities, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devPBest, pBests, sizeof(float) * size, cudaMemcpyHostToDevice);
    cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyHostToDevice);

    // ===== Chronometrage : preparation =====
    cudaEvent_t e0, e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    float ms;
    double t_k1 = 0, t_k2 = 0, t_d2h = 0, t_cpu = 0, t_h2d = 0;
    auto T0 = std::chrono::high_resolution_clock::now();

    for (int iter = 0; iter < MAX_ITER; iter++)
    {
        // (1) kernelUpdateParticle -> t_k1   
        cudaEventRecord(e0);
        kernelUpdateParticle<<<blocksNum, threadsNum>>>(devPos, devVel, devPBest, devGBest,
                                                        getRandomClamped(), getRandomClamped());
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_k1 += ms;

        // (2) kernelUpdatePBest -> t_k2   (meme principe que (1))
        cudaEventRecord(e0);
        kernelUpdatePBest<<<blocksNum, threadsNum>>>(devPos, devPBest, devGBest);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_k2 += ms;

        // (3) copie pBests GPU -> CPU -> t_d2h   (une copie se mesure comme un kernel)
        cudaEventRecord(e0);
        cudaMemcpy(pBests, devPBest, sizeof(float) * size, cudaMemcpyDeviceToHost);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_d2h += ms;

        // (4) boucle CPU du gBest -> t_cpu
        //     C'est du code CPU : on utilise une horloge CPU (std::chrono), pas cudaEvent.
        auto c0 = std::chrono::high_resolution_clock::now();
        for (int i = 0; i < size; i += NUM_OF_DIMENSIONS)
        {
            for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
                temp[k] = pBests[i + k];
            if (host_fitness_function(temp) < host_fitness_function(gBest))
            {
                for (int k = 0; k < NUM_OF_DIMENSIONS; k++)
                    gBest[k] = temp[k];
            }
        }

        auto c1 = std::chrono::high_resolution_clock::now();
        t_cpu += std::chrono::duration<double, std::milli>(c1 - c0).count();

        // (5) copie gBest CPU -> GPU -> t_h2d
        cudaEventRecord(e0);
        cudaMemcpy(devGBest, gBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyHostToDevice);
        cudaEventRecord(e1); cudaEventSynchronize(e1);
        cudaEventElapsedTime(&ms, e0, e1); t_h2d += ms;
    }

    // ===== Chronometrage : resultats =====
    auto T1 = std::chrono::high_resolution_clock::now();
    double total = std::chrono::duration<double, std::milli>(T1 - T0).count();
    double somme = t_k1 + t_k2 + t_d2h + t_cpu + t_h2d;
    printf("\n%-22s %10s %8s\n", "Morceau", "ms", "%");
    printf("%-22s %10.1f %7.1f%%\n", "kernelUpdateParticle", t_k1,  100*t_k1/total);
    printf("%-22s %10.1f %7.1f%%\n", "kernelUpdatePBest",    t_k2,  100*t_k2/total);
    printf("%-22s %10.1f %7.1f%%\n", "memcpy pBests D->H",   t_d2h, 100*t_d2h/total);
    printf("%-22s %10.1f %7.1f%%\n", "boucle CPU gBest",     t_cpu, 100*t_cpu/total);
    printf("%-22s %10.1f %7.1f%%\n", "memcpy gBest H->D",    t_h2d, 100*t_h2d/total);
    printf("%-22s %10.1f %7.1f%%\n", "SOMME des parties",    somme, 100*somme/total);
    printf("%-22s %10.1f\n",         "TOTAL boucle",         total);
    cudaEventDestroy(e0); cudaEventDestroy(e1);

    cudaMemcpy(positions, devPos, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(velocities, devVel, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(pBests, devPBest, sizeof(float) * size, cudaMemcpyDeviceToHost);
    cudaMemcpy(gBest, devGBest, sizeof(float) * NUM_OF_DIMENSIONS, cudaMemcpyDeviceToHost);

    cudaFree(devPos); cudaFree(devVel); cudaFree(devPBest); cudaFree(devGBest);
}


Petit contrôle : on doit trouver **8** `cudaEventRecord` (2 par morceau GPU × 4 morceaux GPU).

In [ ]:
!grep -c cudaEventRecord /content/DE_CUDA/src/pso_timing/kernel.cu

### Cellule 4 — Compiler et exécuter

Tu vas voir des **warnings** en violet (`math_functions.h`, `#546-D`) : ce sont les mêmes qu'à l'étape 1.1, ils sont normaux. Seul un message contenant **`error`** empêcherait la compilation.

Le programme tourne ~5 à 10 s : attends que l'icône arrête de tourner. Le tableau apparaît à la fin.

In [ ]:
%cd /content/DE_CUDA/src/pso_timing
!nvcc -O2 -arch=sm_75 -o pso_timing main.cpp kernel.cpp kernel.cu
!./pso_timing

### Comment lire le résultat

Tu obtiens un tableau `Morceau | ms | %`. Le **%** est la part du temps total prise par chaque morceau.

**Colle-moi ce tableau dans le chat**, puis réponds (même en une phrase chacune) :

1. Quel morceau prend le plus de temps ? Est-ce du **calcul GPU** ?
2. La ligne **SOMME** est-elle proche de 100 % ? S'il manque quelques %, où sont-ils passés ? *(indice : le CPU tire 2 nombres aléatoires et lance les kernels, ce n'est dans aucun morceau)*
3. Le **TOTAL** est-il plus long que les ~3 s de l'étape 1.1 ? Pourquoi ? *(indice : `cudaEventSynchronize` oblige le CPU à attendre 4 fois par itération)*
4. Qu'est-ce que ça t'apprend pour écrire le DE sur GPU ?

### Cellule 5 — Sauvegarder sur ta branche

Ne l'exécute qu'**après** avoir obtenu le tableau.

In [ ]:
%cd /content/DE_CUDA
!git add src/pso_timing/kernel.cu
!git commit -m "1.3 : instrumentation cudaEvent + chrono du PSO (refs #2)"
!git push -u origin amina/1.3-chronometrage

### Dernière étape — la pull request (sur github.com)

1. Va sur github.com/amenacer/DE_CUDA : un bandeau jaune **« amina/1.3-chronometrage had recent pushes »** → **Compare & pull request**.
2. Titre : `1.3 Chronometrage du PSO`. Dans la description, colle ton tableau + tes 4 réponses, et ajoute une ligne `closes #2`.
3. À droite, **Reviewers** → Lounis (s'il a accepté l'invitation).
4. **Create pull request**, puis déplace la carte #2 dans **In Review** sur le tableau du projet.